In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from pyod.models.ecod import ECOD
from pyod.models.cblof import CBLOF
from scipy.stats import spearmanr

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
import sys
sys.path.insert(0, "..")          # le notebook vit dans notebooks/, le paquet a la racine
from ran_aiops.chemins import KPI_TABLE

df = pd.read_csv(KPI_TABLE, skip_blank_lines=True, low_memory=False)
df = df.dropna(how="all").reset_index(drop=True)

kpi_cols = list(df.columns[5:])          # les 9 colonnes KPI
print("Lignes x colonnes :", df.shape)
print("Regions:", df["Region"].nunique(),"| Nodes:", df["Node"].nunique(),"| Cellules:", df["EUtranCell Id"].nunique())
df.head()

In [ ]:
df.dtypes

In [ ]:
for c in kpi_cols:
    if df[c].dtype == object:
        bad = df[c][pd.to_numeric(df[c], errors="coerce").isna() & df[c].notna()]
        if len(bad):
            print(f"{c:50s} {list(bad.unique()[:3])}  ->  {len(bad)} lignes")

In [ ]:
# Conversion en numerique. On garde les #DIV/0 en NaN  et on en garde la trace via un flag .
div0 = pd.DataFrame(False, index=df.index, columns=kpi_cols)
for c in kpi_cols:
    div0[c] = df[c].astype(str).str.contains("DIV", na=False)
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["had_div0_any_kpi"] = div0.any(axis=1)
print("Lignes avec au moins un #DIV/0 :", int(df["had_div0_any_kpi"].sum()))

In [ ]:
# --- Preprocessing des NaN (ex-#DIV/0) ---
df["n_nan_kpi"] = df[kpi_cols].isna().sum(axis=1)
print("KPI manquants par ligne :")
print(df["n_nan_kpi"].value_counts().sort_index())

n_all_nan = int((df["n_nan_kpi"] == len(kpi_cols)).sum())
mask_drop = df["n_nan_kpi"] >= 6
print(f"\nLignes 100% NaN (aucun KPI ce jour-la) : {n_all_nan}")
print(f"Lignes retirees (>=6 KPI manquants)    : {int(mask_drop.sum())}")

n_before = len(df)
df = df[~mask_drop].reset_index(drop=True)
print(f"Lignes : {n_before} -> {len(df)}")

In [ ]:
# Dates, jour de la semaine, week-end, et controle des doublons
df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")
df["day_of_week"] = df["Date"].dt.day_name()
df["is_weekend"] = df["Date"].dt.dayofweek >= 5
print("Periode :", df["Date"].min().date(), "->", df["Date"].max().date())
print("Doublons (cellule, date) :", int(df.duplicated(["EUtranCell Id", "Date"]).sum()))

In [ ]:
missing = pd.DataFrame({
    "n_missing": df[kpi_cols].isna().sum(),
    "pct": (df[kpi_cols].isna().mean() * 100).round(2),
})
missing.sort_values("pct", ascending=False)

In [ ]:
# Completude par cellule :cellule apparaissant un nombre faible de jours -> moyennes moins fiables.
cell_days = df.groupby("EUtranCell Id").size()
print("Cellules totales   :", len(cell_days))
print("Avec 30 jours      :", int((cell_days == 30).sum()))
print("Moins de 25 jours  :", int((cell_days < 25).sum()))
print("Moins de 20 jours  :", int((cell_days < 20).sum()))

In [ ]:
# Statistiques descriptives des 9 KPI
df[kpi_cols].describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]].round(2)

In [ ]:
# Controle de coherence : les KPI en % doivent rester dans [0, 100]
n_oob = 0
for c in kpi_cols:
    oob = int(((df[c] < 0) | (df[c] > 100)).sum())
    n_oob += oob
    if oob:
        print(c, "->", oob, "valeurs hors [0,100]")
print("Total valeurs hors bornes :", n_oob)

In [ ]:
print(df[kpi_cols].skew().round(2))
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for ax, c in zip(axes.ravel(), kpi_cols):
    ax.hist(df[c].dropna(), bins=50, color="blue")
    ax.set_title(c, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution des 9 KPI (skew + histogrammes) tres asymetriques
# taux de succes ecrases vers 100, taux de perte vers 0. Une regle 1.5xIQR ou un
# z-score classique (moyenne/ecart-type) est peu fiable -> on ira vers du robuste.

In [ ]:
# Boxplots des 9 KPI
fig, axes = plt.subplots(3, 3, figsize=(15, 9))
for ax, c in zip(axes.ravel(), kpi_cols):
    ax.boxplot(df[c].dropna(), showfliers=False)
    ax.set_title(c, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
corr = df[kpi_cols].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            xticklabels=[c[:18] for c in kpi_cols],
            yticklabels=[c[:18] for c in kpi_cols])
plt.title("Correlation entre KPI")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation entre KPI : globalement faible. Deux paires ressortent (~0.55) mais
# pas assez pour retirer un KPI -> on garde les 9 comme features.

In [ ]:
test_kpi = "EN_DC_SETUP_succ_RATE_gNB (%)"
order = df.groupby("Region")[test_kpi].median().sort_values().index
plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x="Region", y=test_kpi, order=order, showfliers=False)
plt.xticks(rotation=45, ha="right")
plt.title(f"Distribution regionale - {test_kpi}")
plt.tight_layout()
plt.show()

In [ ]:
# Comparaison par region : les medianes sont quasi identiques (~99%) -> les regions sont homogenes, aucune n'est structurellement avantagee.
# Le choix "baseline par region et non par cellule" vient de l'objectif :  on veut detecter les cellules chroniquement sous leurs pairs. Une baseline par
# cellule (chaque cellule comparee a son propre historique) donnerait z~0 pour une cellule toujours basse -> invisible. La cellule reste l'unite analysee, mais la
# reference est les pairs regionaux, pas son propre historique.

In [ ]:
# Tendance temporelle du reseau : controle de stabilite pour valider la baseline statique.
daily = df.groupby("Date")[kpi_cols].mean()
fig, axes = plt.subplots(3, 3, figsize=(15, 9))
for ax, c in zip(axes.ravel(), kpi_cols):
    ax.plot(daily.index, daily[c], color="green")
    ax.set_title(c, fontsize=8)
    ax.tick_params(axis="x", rotation=45, labelsize=6)
plt.tight_layout()
plt.show()

In [ ]:
# La plupart des KPI oscillent sans tendance (bruit normal). Exception : EN_DC_SETUP_succ_RATE_eNB et Diff_E-RAB_Retainibility derivent
# de ~99% a ~96% et de 0.15% a 0.7% sur le mois -> derive globale du reseau (toutes cellules a la fois), pas une anomalie de cellule,

In [ ]:
# Effet jour de semaine (week-end vs semaine)
# concentration des #DIV/0 par region, a titre de controle.
print(df.groupby("is_weekend")[kpi_cols].mean().round(2).T)
print()
print("Part de lignes #DIV/0 par region (%) :")
print((df.groupby("Region")["had_div0_any_kpi"].mean() * 100).round(2).sort_values(ascending=False))

In [ ]:
reliable = cell_days[cell_days >= 20].index
ref = df[df["EUtranCell Id"].isin(reliable)]

z = pd.DataFrame(index=df.index)
for c in kpi_cols:
    med = df["Region"].map(ref.groupby("Region")[c].median())
    mad = df["Region"].map(ref.groupby("Region")[c].apply(lambda s: (s - s.median()).abs().median()))
    scale = (1.4826 * mad).replace(0, np.nan)      # MAD nulle -> dispersion nulle, z indefini
    z["z_" + c] = (df[c] - med) / scale
z = z.clip(-10, 10)
z_cols = list(z.columns)
df = pd.concat([df, z], axis=1)
print("Cellules fiables pour la baseline (>=20 j) :", len(reliable), "/", len(cell_days))
z[z_cols].describe().T[["mean", "std", "min", "max"]].round(2)

In [ ]:
# Cellules a faible historique (<20 jours) : ecartees de la baseline (peu fiables) mais
# leurs jours restent scores. Gardees a part, jamais etiquetees "chronique".
cell_low = cell_days[cell_days < 20]
print("Cellules a faible historique (<20 j) :", len(cell_low))


# Matrice de features : une ligne = une cellule un jour. z indefinis (KPI absent ou MAD
# nulle) remplaces par 0 = "sur la baseline regionale".
# Note : les jours de silence total (trafic nul -> tous KPI en #DIV/0) ont ete retires en
# amont ; leur detection (panne franche) releverait d'un detecteur d'activite distinct,
# prevu en extension.
D = df[["EUtranCell Id", "Date", "Region"] + z_cols].copy()
X = D[z_cols].fillna(0.0).values
print("Cellules-jours a scorer :", X.shape)

In [ ]:
# --- Modeles non supervises au niveau cellule-jour ---
CONTAMINATION = 0.03
X_scaled = StandardScaler().fit_transform(X)

iforest = IsolationForest(n_estimators=150, contamination=CONTAMINATION, random_state=RANDOM_STATE)
iforest.fit(X_scaled)
score_if = -iforest.score_samples(X_scaled)

pca = PCA(n_components=0.8, random_state=RANDOM_STATE)
pca.fit(X_scaled)
recon = pca.inverse_transform(pca.transform(X_scaled))
score_pca = ((X_scaled - recon) ** 2).sum(axis=1)

ecod = ECOD(contamination=CONTAMINATION).fit(X_scaled)     # objet garde : sert a l'attribution par KPI
score_ecod = ecod.decision_scores_
score_cblof = CBLOF(contamination=CONTAMINATION, n_clusters=8, random_state=RANDOM_STATE).fit(X_scaled).decision_scores_

scores_all = {"IForest": score_if, "PCA": score_pca, "ECOD": score_ecod, "CBLOF": score_cblof}
flags = {n: s >= np.quantile(s, 1 - CONTAMINATION) for n, s in scores_all.items()}
D["score_if"] = score_if
D["score_pca"] = score_pca
D["score_ecod"] = score_ecod
D["score_cblof"] = score_cblof

# --- Detecteur de reference : ECOD ---
# Choix appuye sur les deux validations ci-dessous :
#   detection  ROC-AUC 0.939 / avg-prec 0.296  contre 0.883 / 0.083 pour IForest
#   stabilite  Jaccard 0.975 / Spearman 1.000  contre 0.688 / 0.981
# CBLOF detecte un peu mieux (0.963 / 0.504) mais son Jaccard de 0.456 signifie que
# la moitie de ses anomalies changent d'une execution a l'autre : inutilisable comme
# verite terrain pour les tickets de reclamation, qui figent cette liste.
# ECOD attribue en plus le score par KPI nativement, sans passer par SHAP (cf. plus bas).
MODELE_REF = "ECOD"
flag_ref = flags[MODELE_REF]
D["score_anomalie"] = score_ecod    # nom neutre : la localisation ne doit dependre d'aucun modele
D["flag"] = flag_ref
D["n_models"] = np.sum(list(flags.values()), axis=0)        # nb de modeles qui flaguent la cellule-jour
for n in flags:
    print(f"{n:9s} flagues : {int(flags[n].sum())}")
print("Cellules-jours flaguees par les 4 modeles :", int((D["n_models"] == 4).sum()))
print("Detecteur de reference :", MODELE_REF)

In [ ]:
#  top cellules-jours les plus anormales (IForest).
D["nb_modeles_flag"] = sum(flags.values())
D.sort_values("score_if", ascending=False).head(10)[["EUtranCell Id", "Date", "Region", "score_if", "score_pca", "score_ecod", "score_cblof", "nb_modeles_flag"]]

In [ ]:
# Accord de rang entre les quatre modeles (Spearman).
rank_corr = pd.DataFrame(
    {"IForest": score_if, "PCA": score_pca, "ECOD": score_ecod, "CBLOF": score_cblof}
).corr(method="spearman")
rank_corr.round(2)

In [ ]:
# --- Validation 1 : stabilite par bootstrap ---
# Trop de lignes pour re-entrainer sur tout le jeu -> on evalue la stabilite sur un
# echantillon de cellules-jours, en re-entrainant sur des tirages bootstrap. On mesure la
# stabilite du top-k (Jaccard) et de l'ordre des scores (Spearman).
def jaccard(a, b):
    return len(a & b) / len(a | b)

rng = np.random.RandomState(RANDOM_STATE)
sub = rng.choice(len(X_scaled), 25000, replace=False)
Xs = X_scaled[sub]
k = int(CONTAMINATION * len(Xs))
n_runs = 8
tops = {"IForest": [], "PCA": [], "ECOD": [], "CBLOF": []}
ranks = {"IForest": [], "PCA": [], "ECOD": [], "CBLOF": []}
for b in range(n_runs):
    idx = rng.choice(len(Xs), len(Xs), replace=True)
    Xb = Xs[idx]
    s_if = -IsolationForest(n_estimators=100, contamination=CONTAMINATION, random_state=b).fit(Xb).score_samples(Xs)
    pb = PCA(n_components=0.8, random_state=b).fit(Xb)
    s_pca = ((Xs - pb.inverse_transform(pb.transform(Xs))) ** 2).sum(axis=1)
    s_ecod = ECOD(contamination=CONTAMINATION).fit(Xb).decision_function(Xs)
    s_cblof = CBLOF(contamination=CONTAMINATION, n_clusters=8, random_state=b).fit(Xb).decision_function(Xs)
    for name, s in [("IForest", s_if), ("PCA", s_pca), ("ECOD", s_ecod), ("CBLOF", s_cblof)]:
        tops[name].append(set(np.argsort(s)[::-1][:k]))
        ranks[name].append(s)
for name in tops:
    js = [jaccard(tops[name][i], tops[name][j]) for i in range(n_runs) for j in range(i + 1, n_runs)]
    sp = [spearmanr(ranks[name][i], ranks[name][j]).correlation for i in range(n_runs) for j in range(i + 1, n_runs)]
    print(f"{name:9s} Jaccard top-{k} = {np.mean(js):.3f}   Spearman = {np.mean(sp):.3f}")

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score
naive_flag = (np.abs(X) > 3).any(axis=1)
clean_idx = np.where(~naive_flag)[0]
sub = rng.choice(clean_idx, 30000, replace=False)
Xc = X[sub].copy()
n_inject = 300
inj = rng.choice(len(Xc), n_inject, replace=False)
for i in inj:
    m = rng.randint(1, 3)                          # 1 ou 2 KPI touches
    j = rng.choice(len(z_cols), m, replace=False)
    Xc[i, j] -= rng.uniform(3, 5, size=m)
y = np.zeros(len(Xc)); y[inj] = 1
Xc_s = StandardScaler().fit_transform(Xc)

s_if = -IsolationForest(n_estimators=150, contamination=CONTAMINATION, random_state=RANDOM_STATE).fit(Xc_s).score_samples(Xc_s)
pj = PCA(n_components=0.8, random_state=RANDOM_STATE).fit(Xc_s)
s_pca = ((Xc_s - pj.inverse_transform(pj.transform(Xc_s))) ** 2).sum(axis=1)
s_ecod = ECOD().fit(Xc_s).decision_scores_
s_cblof = CBLOF(n_clusters=8, random_state=RANDOM_STATE).fit(Xc_s).decision_scores_
print(f"{'modele':9s} {'ROC-AUC':>8} {'avg-prec':>9}")
for name, s in [("IForest", s_if), ("PCA", s_pca), ("ECOD", s_ecod), ("CBLOF", s_cblof)]:
    print(f"{name:9s} {roc_auc_score(y, s):8.3f} {average_precision_score(y, s):9.3f}")

In [ ]:
# --- Chronique vs incident ---
# Part de jours anormaux par cellule. Une cellule est CHRONIQUE si elle a assez
# d'historique (>=20 j, sinon on ne peut pas juger une tendance) ET est flaguee sur une
# forte part de ses jours. Les cellules-jours flagues des cellules NON chroniques (et a
# historique suffisant) sont des INCIDENTS ponctuels -> a correler aux reclamations.
MIN_DAYS = 20
SEUIL_CHRONIQUE = 0.5

g = D.groupby("EUtranCell Id")
cell_stat = pd.DataFrame({
    "region": g["Region"].first(),
    "n_days": cell_days,
    "n_flag": g["flag"].sum(),
})
cell_stat["taux_anormal"] = cell_stat["n_flag"] / cell_stat["n_days"]
cell_stat["fiable"] = cell_stat["n_days"] >= MIN_DAYS
cell_stat["is_chronique"] = cell_stat["fiable"] & (cell_stat["taux_anormal"] >= SEUIL_CHRONIQUE)

chroniques = cell_stat[cell_stat["is_chronique"]].copy()
cells_incident = cell_stat.index[cell_stat["fiable"] & ~cell_stat["is_chronique"]]
incidents = D[D["flag"] & D["EUtranCell Id"].isin(cells_incident)].copy()

print("Cellules chroniques         :", len(chroniques))
print("Incidents (cellule-jour)    :", len(incidents))
print("Cellules a faible historique:", len(cell_low))

In [ ]:
# --- Explicabilite : attribution par KPI, native a ECOD ---
# ECOD score chaque dimension separement : O[i, j] = -ln(probabilite de queue) du KPI j
# pour la cellule-jour i, et sum_j O[i, j] = score total. L'attribution est donc EXACTE
# et additive, la ou TreeSHAP n'etait qu'une approximation -- et elle ne depend d'aucun
# modele a arbres, donc plus de dependance a shap.
kpi_names = [c[2:] for c in z_cols]
flag_idx = np.where(flag_ref)[0]
lbl = D.index[flag_idx]

O = np.asarray(ecod.O)
assert np.allclose(O.sum(axis=1), score_ecod), "O doit sommer au score ECOD"
contrib_df = pd.DataFrame(O[flag_idx], index=lbl, columns=kpi_names)

# Un KPI est "en cause" si sa contribution depasse le seuil de queue a 5 % :
# O = -ln(p), donc p < 0.05  <=>  O > -ln(0.05) = 3.0.
# Un incident touche souvent PLUSIEURS KPI : on les liste tous, du plus fort au plus
# faible. Utile car la plainte client peut venir d'un KPI defaillant qui n'est pas LE pire.
SEUIL_KPI = -np.log(0.05)


def kpis_en_cause(row):
    r = row[row > SEUIL_KPI].sort_values(ascending=False)
    if r.empty:                      # aucun KPI au-dela du seuil -> on garde le plus fort
        r = row.sort_values(ascending=False).head(1)
    return "; ".join(r.index)


incidents["kpi_principal"] = contrib_df.idxmax(axis=1).reindex(incidents.index)
incidents["kpis_en_cause"] = contrib_df.apply(kpis_en_cause, axis=1).reindex(incidents.index)

# Chroniques : moyenne des contributions sur les jours flagues -> dominant + liste.
cell_of = D.loc[lbl, "EUtranCell Id"].values
contrib_cell = contrib_df.groupby(cell_of).mean()
chroniques["kpi_dominant"] = contrib_cell.idxmax(axis=1).reindex(chroniques.index)
chroniques["kpis_en_cause"] = contrib_cell.apply(kpis_en_cause, axis=1).reindex(chroniques.index)
print("Seuil de contribution (p<5%) :", round(SEUIL_KPI, 2))
print("Nb moyen de KPI en cause par incident :",
      round(incidents["kpis_en_cause"].str.count(";").add(1).mean(), 1))

In [ ]:
# Vue globale : quels KPI portent les anomalies retenues.
moy = contrib_df.mean().sort_values()
plt.figure(figsize=(9, 5))
moy.plot(kind="barh", color="crimson")
plt.axvline(SEUIL_KPI, ls="--", c="k", lw=1, label="seuil 5 %")
plt.xlabel("contribution ECOD moyenne  (-ln p, plus haut = plus anormal)")
plt.title("Quels KPI portent les anomalies")
plt.legend(); plt.tight_layout(); plt.show()

# Incident le plus fort : contribution de chaque KPI.
top_lbl = incidents.sort_values("score_anomalie", ascending=False).index[0]
contrib = contrib_df.loc[top_lbl].sort_values()
info = D.loc[top_lbl]
plt.figure(figsize=(9, 5))
contrib.plot(kind="barh", color=["red" if v > SEUIL_KPI else "steelblue" for v in contrib])
plt.axvline(SEUIL_KPI, ls="--", c="k", lw=1)
plt.title(f"Contributions ECOD - {info['EUtranCell Id']} le {info['Date'].date()} ({info['Region']})")
plt.xlabel("contribution  (>seuil = KPI en cause)")
plt.tight_layout(); plt.show()
print("Incident le plus fort :", info["EUtranCell Id"], "|", info["Date"].date(),
      "| KPI en cause :", incidents.loc[top_lbl, "kpis_en_cause"])

In [ ]:
from ran_aiops.chemins import ANOMALIES as OUT
OUT.mkdir(parents=True, exist_ok=True)
node_of = df.groupby("EUtranCell Id")["Node"].first()
kpi_mean = df.groupby("EUtranCell Id")[kpi_cols].mean().round(2)

# `score_anomalie` : nom volontairement neutre. Changer de detecteur = reecrire ces CSV
# avec les memes colonnes, sans toucher au code de localisation en aval.

# Incidents : valeurs KPI reelles de la cellule-jour + KPI en cause (contributions ECOD).
inc = df.loc[incidents.index, ["Node", "EUtranCell Id", "Region", "Date"] + kpi_cols].copy()
inc = inc.join(incidents[["score_anomalie", "kpi_principal", "kpis_en_cause"]])
inc.sort_values(["Date", "score_anomalie"], ascending=[True, False]).to_csv(OUT / "anomalies_journalieres.csv", index=False)

# Chroniques : moyenne des KPI reels de la cellule + KPI dominant + Node.
chr_out = chroniques[["region", "n_days", "n_flag", "taux_anormal", "kpi_dominant", "kpis_en_cause"]].copy()
chr_out.insert(0, "Node", node_of.reindex(chr_out.index))
chr_out = chr_out.join(kpi_mean)
chr_out.sort_values("taux_anormal", ascending=False).to_csv(OUT / "cellules_chroniques.csv")

# Faible historique : Node + comptes (jamais etiquetees chroniques).
low = cell_low.rename("n_days").to_frame()
low.insert(0, "Node", node_of.reindex(low.index))
low["n_jours_flagues"] = D.groupby("EUtranCell Id")["flag"].sum().reindex(low.index)
low.to_csv(OUT / "cellules_historique_insuffisant.csv")

print("detecteur :", MODELE_REF)
print("resultats/anomalies/anomalies_journalieres.csv     :", len(inc))
print("resultats/anomalies/cellules_chroniques.csv        :", len(chr_out))
print("resultats/anomalies/cellules_historique_insuffisant:", len(low))